In [2]:
import pandas as pd
import numpy as np
import copy
import json
from typing import Union,TypeVar
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
import scipy.stats as sts
PATH_DATA = 'Data\Sber\ditry'
FILE_NAME = 'transact1_2k'

In [3]:

def remove_outliers(data: pd.DataFrame) -> pd.DataFrame:
    data = copy.deepcopy(data)
    
    for col in data.columns:
        data = data[(data[col] < np.quantile(data[col], 0.75) + 3 * sts.iqr(data[col])) &
                    (data[col] > np.quantile(data[col], 0.25) - 3 * sts.iqr(data[col]))]
    return data

In [4]:
data = pd.read_csv(f'{PATH_DATA}/{FILE_NAME}.csv')

C:\Users\kiril\AppData\Local\Temp\ipykernel_27304\143071573.py:1: DtypeWarning: Columns (15,16) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(f'{PATH_DATA}/{FILE_NAME}.csv')


In [5]:


data.describe()

,CustomerKey,ID,AMOUNT_EQ,MCC,PAY_AMT,CARD,TRAN_METHOD
count,3.238440e+05,3.238440e+05,3.238440e+05,320368.000000,3.235530e+05,3.238440e+05,320381.000000
mean,9.215662e+05,5.925226e+08,2.919939e+03,5643.111035,2.871538e+03,2.930169e+06,65.057375
std,2.669040e+05,1.049492e+08,1.539558e+04,632.396403,1.517068e+04,1.049160e+06,78.875353
min,1.010000e+02,3.941954e+08,1.000000e-02,742.000000,1.000000e-02,4.427000e+03,0.000000
25%,1.006062e+06,5.104213e+08,2.000000e+02,5411.000000,2.000000e+02,2.501038e+06,51.000000
50%,1.007472e+06,5.925989e+08,5.000000e+02,5541.000000,5.000000e+02,3.107585e+06,70.000000
75%,1.009298e+06,6.787873e+08,1.500000e+03,5921.000000,1.484000e+03,3.713112e+06,70.000000
max,1.010759e+06,8.038694e+08,4.200000e+06,9405.000000,4.200000e+06,1.386370e+07,912.000000


In [6]:
# data = data.dropna(subset=["CHANNEL"]).reset_index(drop=True)
data = data.drop(columns=["USED_PAY_SERVICE", "CHANNEL", 'RETAILER'])

In [7]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 323844 entries, 0 to 323843
Data columns (total 20 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   CustomerKey         323844 non-null  int64  
 1   ID                  323844 non-null  int64  
 2   TRANS_TIME          317746 non-null  object 
 3   AMOUNT_EQ           323844 non-null  float64
 4   MCC                 320368 non-null  float64
 5   PAY_AMT             323553 non-null  float64
 6   CurrencyName        323549 non-null  object 
 7   PaymentSystem       323844 non-null  object 
 8   TERMINAL_CODE       323843 non-null  object 
 9   ADDRESS             65559 non-null   object 
 10  DEVICE_TYPE         65435 non-null   object 
 11  TRANS_DETAIL        320392 non-null  object 
 12  NAME                323844 non-null  object 
 13  OpType              323844 non-null  object 
 14  DETAILED_CARD_TYPE  323844 non-null  object 
 15  IS_OWN_TERMINAL     323844 non-nul

In [8]:
data["CurrencyName"].unique()

array(['Евро', 'Рубль', nan, 'Доллар США', 'Злотый', 'Чешская крона'],
      dtype=object)

In [8]:
# data = data.dropna().reset_index(drop=True)
data['IS_OWN_TERMINAL'].unique()

array([False, True, 'false', 'true', 'Visa Cash Back'], dtype=object)

In [9]:
own_terminal_data  = data[(data['IS_OWN_TERMINAL'] == True) | (data['IS_OWN_TERMINAL'] == 'true'))]

SyntaxError: closing parenthesis ')' does not match opening parenthesis '[' (82880956.py, line 1)

In [ ]:
own_terminal_data[own_terminal_data["TERMINAL_CODE"] == "AC090232"].head()

,CustomerKey,ID,TRANS_TIME,AMOUNT_EQ,MCC,PAY_AMT,CurrencyName,PaymentSystem,TERMINAL_CODE,ADDRESS,DEVICE_TYPE,TRANS_DETAIL,NAME,OpType,DETAILED_CARD_TYPE,IS_OWN_TERMINAL,ACCOUNT_ID,CARD,TRAN_METHOD,DATE
2986,1008620,798051750,08:20:33,2000.0,6011.0,2000.0,Рубль,МИР - НСПК,AC090232,"РОССИЯ,197760,г Санкт-Петербург,,г Кронштадт,у...",ATM,"KRONSHTADT,STANYUKOVI, St Petersburg, RU",Выдача наличных ден. средств через банкомат,Снятие наличных,МИР Классическая Зарплатная,True,1433173,3252366,51.0,2019-01-23
3487,1008621,801496650,11:32:09,900.0,6011.0,900.0,Рубль,MasterCard,AC090232,"РОССИЯ,197760,г Санкт-Петербург,,г Кронштадт,у...",ATM,"KRONSHTADT,STANYUKOVI, St Petersburg, RU",Выдача наличных ден. средств через банкомат,Снятие наличных,MasterCard Unembossed,True,1433179,4175256,51.0,2019-01-28
7431,1008620,670985631,19:53:09,600.0,6011.0,600.0,Рубль,МИР - НСПК,AC090232,"РОССИЯ,197760,г Санкт-Петербург,,г Кронштадт,у...",ATM,"KRONSHTADT,STANYUKOVI, St Petersburg, RU",Выдача наличных ден. средств через банкомат,Снятие наличных,МИР Классическая Зарплатная,True,1433173,3252366,51.0,2018-08-12
8172,1009306,574697156,17:43:30,650.0,4816.0,650.0,Рубль,MasterCard,AC090232,"РОССИЯ,,,,г Санкт-Петербург,,,,",NaN,"KRONSHTADT,STANYUKOVI, St Petersburg, RU",Оплата услуг через банкомат,Оплата,MasterCard Unembossed,True,1788441,2615362,51.0,2018-03-03
8228,1008620,666220188,19:48:17,900.0,6011.0,900.0,Рубль,МИР - НСПК,AC090232,"РОССИЯ,197760,г Санкт-Петербург,,г Кронштадт,у...",ATM,"KRONSHTADT,STANYUKOVI, St Petersburg, RU",Выдача наличных ден. средств через банкомат,Снятие наличных,МИР Классическая Зарплатная,True,1433173,3252366,51.0,2018-08-04


In [ ]:
data.head()
# print(len(data["TERMINAL_CODE"].unique()))

,CustomerKey,ID,TRANS_TIME,AMOUNT_EQ,MCC,PAY_AMT,CurrencyName,PaymentSystem,TERMINAL_CODE,ADDRESS,DEVICE_TYPE,TRANS_DETAIL,NAME,OpType,DETAILED_CARD_TYPE,IS_OWN_TERMINAL,ACCOUNT_ID,CARD,TRAN_METHOD,DATE
0,1010693,769529457,19:21:59,515.70,5411.0,6.49,Евро,MasterCard,50254414,NaN,NaN,"AH TO GO 5833, AMSTERDAM ZUI, NL",Оплата товаров/услуг по карте,Оплата,MasterCard Unembossed,False,14798342,4485507,71.0,2019-01-03
1,1009755,769528386,16:35:54,339.39,4121.0,339.39,Рубль,MasterCard,TwiTerm,NaN,NaN,"UBER TRIP XYKOV HELP., help.uber.com, NL",Оплата товаров/услуг по карте,Оплата,MasterCard World,False,1710442,2678444,12.0,2019-01-03
2,1009755,769531449,13:40:41,210.60,8111.0,210.60,Рубль,MasterCard,00746775,NaN,NaN,"TERM FLUVIAL CAIS SODR, LISBOA, PT",Оплата товаров/услуг по карте,Оплата,MasterCard World,False,1710442,2678444,51.0,2019-01-03
3,1006571,769523812,11:31:26,461.70,5462.0,461.70,Рубль,MasterCard,27052300,NaN,NaN,"MOULIN PLEYEL, ST DENIS, FR",Оплата товаров/услуг по карте,Оплата,MasterCard Unembossed,False,7638425,4477020,70.0,2019-01-03
4,1009886,769531319,11:36:29,10637.73,5411.0,10637.73,Рубль,MasterCard,39685FCE,NaN,NaN,"S bra Prisma, TartuViro, EE",Оплата товаров/услуг по карте,Оплата,MasterCard World,False,1434868,3459970,71.0,2019-01-03


In [ ]:

own_terminal_data.info()
own_terminal_data.describe()

NameError: name 'own_terminal_data' is not defined

TERMINAL_CODE - много единичных значений можно выкинуть или оставить. Также смущает TwiTerm или NONE.

In [ ]:
own_terminal_data_nan  = own_terminal_data[own_terminal_data['ADDRESS'].isnull()]

In [ ]:
own_terminal_data_nan.head()

,CustomerKey,ID,TRANS_TIME,AMOUNT_EQ,MCC,PAY_AMT,CurrencyName,PaymentSystem,TERMINAL_CODE,ADDRESS,DEVICE_TYPE,TRANS_DETAIL,NAME,OpType,DETAILED_CARD_TYPE,IS_OWN_TERMINAL,ACCOUNT_ID,CARD,TRAN_METHOD,DATE
14,1010229,768028014,18:43:08,155.55,4814.0,155.55,Рубль,МИР - НСПК,AC090226,NaN,NaN,"NOVATOROV, 1, St Petersburg, RU",Оплата услуг через банкомат,Оплата,МИР Классическая Зарплатная,True,7043888,3683363,51.0,2019-01-03
423,1005654,800225569,11:19:23,1200.00,4814.0,1200.00,Рубль,MasterCard,AC090161,NaN,NaN,"MOSKOVSKIY, 141, St Petersburg, RU",Оплата услуг через банкомат,Оплата,MasterCard Unembossed,True,10552018,3029503,51.0,2019-01-26
424,1006896,803404948,17:31:47,1000.00,4814.0,1000.00,Рубль,MasterCard,AC090355,NaN,NaN,"MOSKOVSKIY, 28, St Petersburg, RU",Оплата услуг через банкомат,Оплата,MasterCard Unembossed,True,1430771,4180445,51.0,2019-01-31
425,101017,803354186,19:04:09,1000.00,4814.0,1000.00,Рубль,МИР - НСПК,AC090622,NaN,NaN,"UL.BELY KUNA,3-A, St Petersburg, RU",Оплата услуг через банкомат,Оплата,МИР Классическая Зарплатная,True,2079143,3751162,51.0,2019-01-31
426,1006651,798050760,08:12:44,700.00,4814.0,700.00,Рубль,МИР - НСПК,AC090032,NaN,NaN,"HERSONSKIY PROEZD, 2, St Petersburg, RU",Оплата услуг через банкомат,Оплата,МИР Классическая Зарплатная,True,2376625,3701208,51.0,2019-01-23


In [ ]:
data['TERMINAL_CODE'].value_counts()
print(data['TERMINAL_CODE'].value_counts())


TERMINAL_CODE
TwiTerm     4132
11436154    2226
NONE        1765
10000001    1079
00000001     777
            ... 
11385588       1
35730001       1
20375139       1
20454366       1
IDM22870       1
Name: count, Length: 56727, dtype: int64


Удаляем столбцы без значений или с пропусками

CHANNEL - Возмодно как-то считается или заполняется, но зависимости не заметил

ADDRESS - решено пока удалить, возмодно как0то получить из TRANS_DETAIL

Не понял что такое RATEILER

У USED_PAY_SERVICE есть зависимость с типом операции, как правило с непустыми значениями.

In [ ]:
data['USED_PAY_SERVICE'].value_counts()
print(data['USED_PAY_SERVICE'].value_counts())

KeyError: 'USED_PAY_SERVICE'

In [ ]:
data = data.drop([ 'PAY_AMT' , 'CurrencyName', 'DEVICE_TYPE', 'ADDRESS', 'CHANNEL', 'RETAILER', 'USED_PAY_SERVICE'], axis=1)

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 323844 entries, 0 to 323843
Data columns (total 16 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   CustomerKey         323844 non-null  int64  
 1   ID                  323844 non-null  int64  
 2   TRANS_TIME          317746 non-null  object 
 3   AMOUNT_EQ           323844 non-null  float64
 4   MCC                 320368 non-null  float64
 5   PaymentSystem       323844 non-null  object 
 6   TERMINAL_CODE       323843 non-null  object 
 7   TRANS_DETAIL        320392 non-null  object 
 8   NAME                323844 non-null  object 
 9   OpType              323844 non-null  object 
 10  DETAILED_CARD_TYPE  323844 non-null  object 
 11  IS_OWN_TERMINAL     323844 non-null  object 
 12  ACCOUNT_ID          323844 non-null  object 
 13  CARD                323844 non-null  int64  
 14  TRAN_METHOD         320381 non-null  float64
 15  DATE                323844 non-nul

In [ ]:
data = data.dropna().reset_index(drop=True)

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 314339 entries, 0 to 314338
Data columns (total 16 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   CustomerKey         314339 non-null  int64  
 1   ID                  314339 non-null  int64  
 2   TRANS_TIME          314339 non-null  object 
 3   AMOUNT_EQ           314339 non-null  float64
 4   MCC                 314339 non-null  float64
 5   PaymentSystem       314339 non-null  object 
 6   TERMINAL_CODE       314339 non-null  object 
 7   TRANS_DETAIL        314339 non-null  object 
 8   NAME                314339 non-null  object 
 9   OpType              314339 non-null  object 
 10  DETAILED_CARD_TYPE  314339 non-null  object 
 11  IS_OWN_TERMINAL     314339 non-null  object 
 12  ACCOUNT_ID          314339 non-null  object 
 13  CARD                314339 non-null  int64  
 14  TRAN_METHOD         314339 non-null  float64
 15  DATE                314339 non-nul

В общем зависимость такая. Везде где пропущены MCC, то скорее  всего это Внесение наличных. При жтом если заполнен адрес, то это TRAN_METHOD = 51

In [ ]:
data = data.rename(columns={'IS_OWN_TERMINAL': 'ISOWNTERMINAL'})

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 314339 entries, 0 to 314338
Data columns (total 16 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   CustomerKey         314339 non-null  int64  
 1   ID                  314339 non-null  int64  
 2   TRANS_TIME          314339 non-null  object 
 3   AMOUNT_EQ           314339 non-null  float64
 4   MCC                 314339 non-null  float64
 5   PaymentSystem       314339 non-null  object 
 6   TERMINAL_CODE       314339 non-null  object 
 7   TRANS_DETAIL        314339 non-null  object 
 8   NAME                314339 non-null  object 
 9   OpType              314339 non-null  object 
 10  DETAILED_CARD_TYPE  314339 non-null  object 
 11  ISOWNTERMINAL       314339 non-null  object 
 12  ACCOUNT_ID          314339 non-null  object 
 13  CARD                314339 non-null  int64  
 14  TRAN_METHOD         314339 non-null  float64
 15  DATE                314339 non-nul

In [ ]:
data['MCC'] = data['MCC'].astype(int).astype(str)
data['ACCOUNT_ID'] = data['ACCOUNT_ID'].astype(int).astype(str)
data['CARD'] = data['CARD'].astype(int).astype(str)
data['CustomerKey'] = data['CustomerKey'].astype(int).astype(str)
data['ID'] = data['ID'].astype(int).astype(str)
data['TRAN_METHOD'] = data['TRAN_METHOD'].astype(int).astype(str)

In [ ]:
data = data.loc[remove_outliers(data[['AMOUNT_EQ']]).index]

In [ ]:
data['DATE'] = pd.to_datetime(data['DATE'])
data['TRANS_TIME'] = pd.to_datetime(data['TRANS_TIME'])

C:\Users\kiril\AppData\Local\Temp\ipykernel_16404\3826381048.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data['TRANS_TIME'] = pd.to_datetime(data['TRANS_TIME'])


In [ ]:
print(data)

       CustomerKey         ID          TRANS_TIME  AMOUNT_EQ   MCC  \
0          1010693  769529457 2025-11-04 19:21:59     515.70  5411   
1          1009755  769528386 2025-11-04 16:35:54     339.39  4121   
2          1009755  769531449 2025-11-04 13:40:41     210.60  8111   
3          1006571  769523812 2025-11-04 11:31:26     461.70  5462   
5          1009886  769533934 2025-11-04 16:18:32    2407.32  5541   
...            ...        ...                 ...        ...   ...   
314333     1008881  597346536 2025-11-04 17:29:37     400.00  6011   
314334     1006938  597354435 2025-11-04 17:28:23     400.00  6011   
314336     1006818  597140488 2025-11-04 14:19:22    2000.00  6011   
314337     1005092  597304465 2025-11-04 13:41:47    4000.00  6011   
314338     1010578  596968761 2025-11-04 19:23:41    1000.00  6011   

       PaymentSystem TERMINAL_CODE                              TRANS_DETAIL  \
0         MasterCard      50254414          AH TO GO 5833, AMSTERDAM ZUI, NL   

In [ ]:
data.to_csv('Data/transactions_first_pert.csv', index=False)